In [0]:
#read from silver
from pyspark.sql import functions as F

base = "/Volumes/workspace/test_pipeline/test_pipeline_volume"

silver = spark.read.parquet(f"{base}/silver/orders_customers")

In [0]:
#aggregate
gold = (
    silver.groupBy("customer_state")
    .agg(
        F.count("*").alias("total_orders"),
        F.sum(F.when(F.col("order_status") == "delivered", 1).otherwise(0)).alias("delivered_orders"),
        F.round(F.avg("delivery_days"), 2).alias("avg_delivery_days"),
    )
    .orderBy(F.col("total_orders").desc())
)
display(gold)

In [0]:
gold.write.mode("overwrite").parquet(f"{base}/gold/orders_by_state")
print("Gold written:", gold.count())